# Data Extraction: Bitcoin, S&P 500, and Gold
Raw OHLCV data from Yahoo Finance — no imputation or forward-fill.

In [22]:
import yfinance as yf
import pandas as pd
import time

In [23]:
# BTC-USD : Bitcoin in USD
# ^GSPC   : S&P 500 Index
# GC=F    : Gold Futures (front-month)
TICKERS = {
    "BTC": "BTC-USD",
    "SP500": "^GSPC",
    "Gold": "GC=F",
}

START = "2017-01-01"
END   = "2026-03-01"

In [24]:
raw = {}
for name, ticker in TICKERS.items():
    print(f"Downloading {name} ({ticker})...")
    for attempt in range(1, 6):
        df = yf.download(ticker, start=START, end=END, auto_adjust=True, progress=False)
        if not df.empty:
            break
        wait = attempt * 10
        print(f"  Rate limited - waiting {wait}s (attempt {attempt}/5)...")
        time.sleep(wait)
    if df.empty:
        print(f"  FAILED to download {name} after 5 attempts.")
        continue
    df.index = pd.to_datetime(df.index)
    raw[name] = df
    print(f"  {len(df)} rows  |  {df.index[0].date()} -> {df.index[-1].date()}")
    time.sleep(2)

  3346 rows  |  2017-01-01 -> 2026-02-28
  2301 rows  |  2017-01-03 -> 2026-02-27
  2302 rows  |  2017-01-03 -> 2026-02-27


In [25]:
# Preview each raw DataFrame
for name, df in raw.items():
    print(f"\n--- {name} ---")
    print(df.shape)
    print(df.isna().sum())
    display(df.tail())


--- BTC ---
(3346, 5)
Price   Ticker 
Close   BTC-USD    0
High    BTC-USD    0
Low     BTC-USD    0
Open    BTC-USD    0
Volume  BTC-USD    0
dtype: int64


Price,Close,High,Low,Open,Volume
Ticker,BTC-USD,BTC-USD,BTC-USD,BTC-USD,BTC-USD
Date,,,,,
2026-02-24,64080.042969,64992.156250,62553.187500,64616.015625,40849331086
2026-02-25,67960.125000,69953.531250,63942.484375,64077.769531,53629234355
2026-02-26,67453.773438,68843.351562,66523.734375,67954.867188,42988597523
2026-02-27,65881.796875,68220.406250,64946.035156,67456.515625,40283655942
2026-02-28,66995.859375,67714.523438,63062.218750,65878.929688,42041497112



--- SP500 ---
(2301, 5)
Price   Ticker
Close   ^GSPC     0
High    ^GSPC     0
Low     ^GSPC     0
Open    ^GSPC     0
Volume  ^GSPC     0
dtype: int64


Price,Close,High,Low,Open,Volume
Ticker,^GSPC,^GSPC,^GSPC,^GSPC,^GSPC
Date,,,,,
2026-02-23,6837.750000,6916.959961,6819.819824,6901.250000,5638350000
2026-02-24,6890.069824,6899.169922,6815.430176,6837.370117,5266090000
2026-02-25,6946.129883,6952.509766,6915.149902,6915.149902,5328060000
2026-02-26,6908.859863,6947.250000,6859.729980,6944.740234,5889550000
2026-02-27,6878.879883,6882.959961,6831.740234,6856.540039,6665660000



--- Gold ---
(2302, 5)
Price   Ticker
Close   GC=F      0
High    GC=F      0
Low     GC=F      0
Open    GC=F      0
Volume  GC=F      0
dtype: int64


Price,Close,High,Low,Open,Volume
Ticker,GC=F,GC=F,GC=F,GC=F,GC=F
Date,,,,,
2026-02-23,5204.700195,5211.600098,5120.299805,5120.299805,779
2026-02-24,5155.799805,5159.000000,5112.700195,5158.799805,88
2026-02-25,5206.399902,5206.399902,5166.000000,5166.000000,1772
2026-02-26,5176.500000,5199.200195,5143.899902,5177.200195,1520
2026-02-27,5230.500000,5280.000000,5176.700195,5186.700195,354


In [26]:
for name, df in raw.items():
    print(f"\n--- {name} ---")
    display(df['Close'].describe())


--- BTC ---


Ticker,BTC-USD
count,3346.000000
mean,34401.642485
std,32485.446926
min,777.757019
25%,8154.480347
50%,23004.603516
75%,54824.556641
max,124752.531250



--- SP500 ---


Ticker,^GSPC
count,2301.000000
mean,4007.192706
std,1275.989965
min,2237.399902
25%,2852.500000
50%,3928.860107
75%,4688.680176
max,6978.600098



--- Gold ---


Ticker,GC=F
count,2302.000000
mean,1947.663204
std,770.418696
min,1160.400024
25%,1333.450012
50%,1799.299988
75%,2011.725037
max,5318.399902


In [28]:
import os
os.makedirs("../data", exist_ok=True)

for name, df in raw.items():
    out = f"../data/raw_{name.lower()}.csv"
    df.to_csv(out)
    print(f"Saved {out}")

Saved ../data/raw_btc.csv
Saved ../data/raw_sp500.csv
Saved ../data/raw_gold.csv
